###load_struct_to_prep_orders
####Builds the Gold layer orders fact table (ecom_prep.orders) - joins Silver orders, order items, payments, and products; computes delivery_days, delivery_delay_days, and total_revenue

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from datetime import date

In [0]:
struct_orders = spark.table("ecom_struct.orders")
struct_order_items = spark.table("ecom_struct.order_items")
struct_sellers = spark.table("ecom_struct.sellers")
struct_products = spark.table("ecom_ref_struct.products")
struct_payments = spark.table("ecom_struct.order_payments")

In [0]:
# payments can have multiple rows per order (installments)
# aggregate to one row per order
struct_payments_agg = struct_payments.groupBy("order_id").agg(
    sum("payment_value").alias("payment_value"),
    max("payment_installments").alias("payment_installments"),
    collect_set("payment_type").alias("payment_types")
)

struct_payments_agg = struct_payments_agg.withColumn(
    "payment_type", 
    concat_ws(",", col("payment_types"))
).drop("payment_types")

In [0]:
orders = struct_order_items \
    .join(struct_orders, "order_id", "left") \
    .join(struct_sellers, "seller_id", "left") \
    .join(struct_products, "product_id", "left") \
    .join(struct_payments_agg, "order_id", "left") \
    .select(
        col("order_id"),
        col("order_item_id"),
        col("customer_id"),
        col("seller_id"),
        col("seller_city"),
        col("seller_state"),
        col("product_id"),
        col("product_category_name_english").alias("category_name_english"),
        col("order_status"),
        col("price"),
        col("freight_value"),
        round(col("price") + col("freight_value"), 2).alias("total_revenue"),
        col("payment_value"),
        col("payment_installments"),
        col("payment_type"),
        datediff(
            col("order_delivered_customer_date"),
            col("order_purchase_timestamp")
        ).alias("delivery_days"),
        datediff(
            col("order_estimated_delivery_date"),
            col("order_purchase_timestamp")
        ).alias("estimated_delivery_days"),
        datediff(
            col("order_delivered_customer_date"),
            col("order_estimated_delivery_date")
        ).alias("delivery_delay_days"),
        to_date(col("order_purchase_timestamp")).alias("order_purchase_date")
    )

In [0]:
orders.createOrReplaceTempView("orders")

orders_view = spark.sql('''
    SELECT
        o.order_id,
        o.order_item_id,
        c.customer_key,
        c.customer_unique_id,
        o.seller_id,
        o.seller_city,
        o.seller_state,
        o.product_id,
        o.category_name_english,
        o.order_status,
        o.price,
        o.freight_value,
        o.total_revenue,
        o.payment_value,
        o.payment_installments,
        o.payment_type,
        o.delivery_days,
        o.estimated_delivery_days,
        o.delivery_delay_days,
        o.order_purchase_date
    FROM orders o
    LEFT JOIN ecom_prep.customers_map m ON o.customer_id = m.customer_id
    LEFT JOIN ecom_prep.customers c ON m.customer_unique_id = c.customer_unique_id AND c.is_current = true
''')
orders_view.createOrReplaceTempView("orders_view")

In [0]:
prep_orders = orders_view.select(
    col("order_id"),
    col("order_item_id"),
    col("customer_key"),
    col("customer_unique_id"),
    col("seller_id"),
    col("seller_city"),
    col("seller_state"),
    col("product_id"),
    col("category_name_english"),
    col("order_status"),
    col("price"),
    col("freight_value"),
    col("total_revenue"),
    col("payment_value"),
    col("payment_installments"),
    col("payment_type"),
    col("delivery_days"),
    col("estimated_delivery_days"),
    col("delivery_delay_days"),
    col("order_purchase_date")
)

prep_orders.createOrReplaceTempView("prep_orders")

In [0]:
spark.sql('''
    MERGE INTO ecom_prep.orders AS gold
    USING prep_orders AS updates
    ON gold.order_id = updates.order_id
    AND gold.order_item_id = updates.order_item_id
    WHEN MATCHED THEN UPDATE SET
        gold.customer_key = updates.customer_key,
        gold.customer_unique_id = updates.customer_unique_id,
        gold.seller_id = updates.seller_id,
        gold.seller_city = updates.seller_city,
        gold.seller_state = updates.seller_state,
        gold.product_id = updates.product_id,
        gold.category_name_english = updates.category_name_english,
        gold.order_status = updates.order_status,
        gold.price = updates.price,
        gold.freight_value = updates.freight_value,
        gold.total_revenue = updates.total_revenue,
        gold.payment_value = updates.payment_value,
        gold.payment_installments = updates.payment_installments,
        gold.payment_type = updates.payment_type,
        gold.delivery_days = updates.delivery_days,
        gold.estimated_delivery_days = updates.estimated_delivery_days,
        gold.delivery_delay_days = updates.delivery_delay_days,
        gold.order_purchase_date = updates.order_purchase_date
    WHEN NOT MATCHED THEN INSERT (
        order_id, order_item_id, customer_key, customer_unique_id, seller_id, seller_city, seller_state, product_id, category_name_english, order_status, price, freight_value, total_revenue, payment_value, payment_installments, payment_type, delivery_days, estimated_delivery_days, delivery_delay_days, order_purchase_date
    ) VALUES (
        updates.order_id, updates.order_item_id, updates.customer_key, updates.customer_unique_id, updates.seller_id, updates.seller_city, updates.seller_state, updates.product_id, updates.category_name_english, updates.order_status, updates.price, updates.freight_value, updates.total_revenue, updates.payment_value, updates.payment_installments, updates.payment_type, updates.delivery_days, updates.estimated_delivery_days, updates.delivery_delay_days, updates.order_purchase_date
    )
''').display()